In [7]:
import os
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle

# ── Load PDFs ─────────────────────────────────────────────────────────────────
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

print("Loading PDFs...")
ema_text = extract_text_from_pdf("ema_guideline.pdf")
fda_offlabel_text = extract_text_from_pdf("fda_offlabel.pdf")
fda_fraud_text = extract_text_from_pdf("fda_fraud.pdf")

print(f"EMA Pharmacovigilance Guideline: {len(ema_text):,} characters")
print(f"FDA Off-Label Guidance: {len(fda_offlabel_text):,} characters")
print(f"FDA Fraud Compliance Program: {len(fda_fraud_text):,} characters")

# ── Chunk documents ───────────────────────────────────────────────────────────
def chunk_text(text, source, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append({"text": chunk, "source": source})
    return chunks

ema_chunks = chunk_text(ema_text, "EMA Guideline on Pharmacovigilance")
fda_offlabel_chunks = chunk_text(fda_offlabel_text, "FDA Guidance on Off-Label Use")
fda_fraud_chunks = chunk_text(fda_fraud_text, "FDA Compliance Program on Drug Fraud")
all_chunks = ema_chunks + fda_offlabel_chunks + fda_fraud_chunks

print(f"\nTotal chunks created: {len(all_chunks)}")
print(f"  EMA chunks: {len(ema_chunks)}")
print(f"  FDA Off-Label chunks: {len(fda_offlabel_chunks)}")
print(f"  FDA Fraud chunks: {len(fda_fraud_chunks)}")

Loading PDFs...
EMA Pharmacovigilance Guideline: 567,404 characters
FDA Off-Label Guidance: 94,924 characters
FDA Fraud Compliance Program: 28,525 characters

Total chunks created: 220
  EMA chunks: 179
  FDA Off-Label chunks: 31
  FDA Fraud chunks: 10


In [8]:
# Create embeddings 
print("Loading embedding model... (first time may take a few minutes)")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

texts = [chunk['text'] for chunk in all_chunks]
sources = [chunk['source'] for chunk in all_chunks]

print("Creating embeddings...")
embeddings = embedder.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

print(f"\nEmbeddings shape: {embeddings.shape}")

# Build FAISS index 
print("\nBuilding FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"FAISS index built: {index.ntotal} vectors indexed")

# Save index and metadata 
faiss.write_index(index, "pharmaguard_index.faiss")

with open("pharmaguard_chunks.pkl", "wb") as f:
    pickle.dump({"texts": texts, "sources": sources}, f)

print("\nSaved: pharmaguard_index.faiss")
print("Saved: pharmaguard_chunks.pkl")

Loading embedding model... (first time may take a few minutes)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Creating embeddings...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Embeddings shape: (220, 384)

Building FAISS index...
FAISS index built: 220 vectors indexed

Saved: pharmaguard_index.faiss
Saved: pharmaguard_chunks.pkl


In [9]:
# Test RAG retrieval 
def retrieve_relevant_chunks(query, index, texts, sources, embedder, top_k=3):
    query_embedding = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "text": texts[idx],
            "source": sources[idx],
            "distance": distances[0][i]
        })
    return results

# Test queries simulating fraud scenarios
test_queries = [
    "off-label prescription reporting requirements pharmaceutical company",
    "abnormal prescription volume suspicious prescriber reporting",
    "fraud detection reimbursement claims pharmacovigilance"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    print("-" * 60)
    results = retrieve_relevant_chunks(query, index, texts, sources, embedder)
    for i, r in enumerate(results):
        print(f"Result {i+1} [{r['source']}] (distance: {r['distance']:.3f})")
        print(f"{r['text'][:200]}...")
        print()


Query: off-label prescription reporting requirements pharmaceutical company
------------------------------------------------------------
Result 1 [FDA Guidance on Off-Label Use] (distance: 0.743)
143 use in the FDA-required labeling of an approved/cleared medical product (as that term is 144 defined in this guidance). 145 146 • The term FDA-required labeling includes, but is not necessarily li...

Result 2 [EMA Guideline on Pharmacovigilance] (distance: 0.757)
Where the potential for off-label use has been identified for a product and such use could raise a safety concern (i.e. because there is a justified supposition that an important potential risk might ...

Result 3 [EMA Guideline on Pharmacovigilance] (distance: 0.774)
group of patients;  a different route or method of administration;  a different posology. Guideline on good pharmacovigilance practices (GVP) – Module VI (Rev 2) EMA/873138/2011 Rev 2 Track-change v...


Query: abnormal prescription volume suspicious prescriber r

In [10]:
import pandas as pd
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load results and RAG components 
df_results = pd.read_csv("pharmaguard_results.csv")
flagged = df_results[df_results['predicted_fraud'] == 1].head(10).copy()

index = faiss.read_index("pharmaguard_index.faiss")
with open("pharmaguard_chunks.pkl", "rb") as f:
    data = pickle.load(f)
texts = data['texts']
sources = data['sources']

embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Build query from anomaly 
def build_query(row):
    fraud_type = row['fraud_type']
    drug = row['generic_name']
    specialty = row['specialty']
    
    if fraud_type == 'volume_fraud':
        return f"abnormal high prescription volume reporting requirements {drug}"
    elif fraud_type == 'cost_fraud':
        return f"abnormal reimbursement cost fraud pharmaceutical {drug}"
    elif fraud_type == 'off_label':
        return f"off-label prescription {drug} {specialty} reporting requirements"
    else:
        return f"suspicious prescription pattern {drug}"

# Retrieve regulation for each flagged case 
def retrieve_top(query, top_k=1):
    q_emb = embedder.encode([query]).astype('float32')
    distances, indices = index.search(q_emb, top_k)
    idx = indices[0][0]
    return texts[idx][:400], sources[idx]

print("Linking anomalies to regulatory references...\n")
flagged['query'] = flagged.apply(build_query, axis=1)
flagged[['regulation_text', 'regulation_source']] = flagged['query'].apply(
    lambda q: pd.Series(retrieve_top(q))
)

# Preview 
for _, row in flagged.iterrows():
    print(f"Prescriber: {row['prescriber_id']} | Drug: {row['generic_name']}")
    print(f"Fraud Type: {row['fraud_type']} | Anomaly Score: {row['anomaly_score']:.4f}")
    print(f"Regulation [{row['regulation_source']}]:")
    print(f"{row['regulation_text'][:250]}...")
    print("-" * 70)

# Save
flagged.to_csv("pharmaguard_flagged.csv", index=False)
print("\nSaved: pharmaguard_flagged.csv")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Linking anomalies to regulatory references...

Prescriber: 1811434137 | Drug: Rsvpref3 Antigen/As01e/Pf
Fraud Type: legitimate | Anomaly Score: -0.0534
Regulation [EMA Guideline on Pharmacovigilance]:
quoted as final) Page 61/225 3.2. Where the primary source reports a suspect or interacting branded/proprietary medicinal product name without indicating the pharmaceutical form/presentation of the product and where the proprietary/branded medicinal ...
----------------------------------------------------------------------
Prescriber: 1457350902 | Drug: Alprazolam
Fraud Type: cost_fraud | Anomaly Score: -0.2193
Regulation [FDA Compliance Program on Drug Fraud]:
potential, emerging, and active investigations. Health fraud products that pose a direct health hazard to the user are the agency’s highest health fraud enforcement priority. For the purposes of this compliance program, a product presents a direct he...
----------------------------------------------------------------------
Prescrib